In [1]:
# Cell 1: setup, data, configuration

import glob
import json
import math
import random
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, flush = True)

def find_file(file_name):
    matches = [path for path in glob.glob("/kaggle/input/**/" + file_name, recursive = True) if "__MACOSX" not in path]
    return matches[0]

target_values = pd.read_csv(find_file("student_train.csv"))["value"].to_numpy(dtype = np.float32)
external_values = pd.read_csv(find_file("optional_external_data.csv")).drop(columns = ["time_idx"]).to_numpy(dtype = np.float32)
train_length = len(target_values)
total_length = len(external_values)
assert train_length == 43656 and total_length == 43824

prediction_length = 168
label_length = 84
maximum_input_length = 1008 # All input lengths share the same training targets because windows start here
validation_block_count = 24
validation_start = train_length - validation_block_count * prediction_length
seed_list = [0, 1, 2]
batch_size = 64
training_stride = 4 # Each epoch uses every 4th window with a random offset
head_count = 4
moving_average_kernel = 25
delay_factor = 3
dropout_rate = 0.1
output_directory = "/kaggle/working/"

# External columns: 0 to 5 are continuous (A to F), 6 to 9 are the binary group (G to J)
feature_groups = {
    "none": [],
    "all": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    "continuous": [0, 1, 2, 3, 4, 5],
    "binary": [6, 7, 8, 9],
    "ABC": [0, 1, 2],
    "DEF": [3, 4, 5],
}

# Each placement says where covariates enter, and whether the horizon values are visible
placement_settings = {
    "encoder_only": {"encoder": True, "decoder": False, "future": True, "concatenate": False},
    "decoder_only": {"encoder": False, "decoder": True, "future": True, "concatenate": False},
    "both": {"encoder": True, "decoder": True, "future": True, "concatenate": False},
    "past_only_both": {"encoder": True, "decoder": True, "future": False, "concatenate": False},
    "both_concat": {"encoder": True, "decoder": True, "future": True, "concatenate": True},
}

# Each size is (width, encoder layers, decoder layers)
size_settings = {
    "tiny": (8, 1, 1),
    "small": (16, 1, 1),
    "medium": (32, 1, 1),
    "large": (64, 2, 1),
    "xlarge": (128, 2, 2),
}

# Each mechanism is (decomposition, auto-correlation, decoder trend accumulation)
mechanism_settings = {
    "full": (True, True, True),
    "no_decomposition": (False, True, True),
    "no_autocorrelation": (True, False, True),
    "neither": (False, False, True),
    "no_trend_accumulation": (True, True, False),
}

default_config = {
    "feature_group": "none",
    "placement": "both",
    "size_name": "medium",
    "maximum_epochs": 6,
    "mechanism": "full",
    "loss_name": "mse",
    "input_length": 336,
    "learning_rate": 1e-3,
}
best_config = dict(default_config)

validation_origins = [validation_start + block * prediction_length for block in range(validation_block_count)]
actual_blocks = np.stack([target_values[origin:origin + prediction_length] for origin in validation_origins])
print("train length:", train_length, "validation start:", validation_start, flush = True)

device: cuda
train length: 43656 validation start: 39624


In [2]:
# Cell 2: Autoformer model

class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size

    def forward(self, x):                       # [B,L,C] -> (seasonal, trend)
        half_width = self.kernel_size // 2
        front_padding = x[:, :1, :].repeat(1, half_width, 1) # Repeats the first value
        end_padding = x[:, -1:, :].repeat(1, half_width, 1) # Repeats the last value
        padded = torch.cat([front_padding, x, end_padding], dim = 1)
        trend = F.avg_pool1d(padded.transpose(1, 2), kernel_size = self.kernel_size, stride = 1).transpose(1, 2)
        return x - trend, trend

class IdentitySplit(nn.Module):
    def forward(self, x):
        return x, torch.zeros_like(x) # No decomposition: everything stays in the seasonal stream

def create_split(kernel_size, use_decomposition):
    if use_decomposition:
        return SeriesDecomposition(kernel_size)
    return IdentitySplit()

class AutoCorrelation(nn.Module):
    def __init__(self, factor):
        super().__init__()
        self.factor = factor

    def forward(self, queries, keys, values):   # each [B,L,H,E]
        batch, length, heads, channels = queries.shape
        source_length = values.shape[1]
        if source_length > length:
            values = values[:, -length:] # Keeps the latest part of a longer source
            keys = keys[:, -length:]
        elif source_length < length:
            zeros = torch.zeros(batch, length - source_length, heads, channels, device = values.device)
            values = torch.cat([values, zeros], dim = 1)
            keys = torch.cat([keys, zeros], dim = 1)
        queries = queries.permute(0, 2, 3, 1) # [B,H,E,L]
        keys = keys.permute(0, 2, 3, 1)
        values = values.permute(0, 2, 3, 1)
        spectrum = torch.fft.rfft(queries, dim = -1) * torch.fft.rfft(keys, dim = -1).conj()
        correlation = torch.fft.irfft(spectrum, n = length, dim = -1) # Circular correlation for every delay
        top_k = max(1, int(self.factor * math.log(length)))
        scores, delays = torch.topk(correlation.mean(dim = (1, 2)), top_k, dim = -1) # [B,K]
        weights = torch.softmax(scores, dim = -1)
        positions = torch.arange(length, device = values.device)
        output = torch.zeros_like(values)
        for delay_index in range(top_k):
            source_positions = (positions.unsqueeze(0) - delays[:, delay_index].unsqueeze(1)) % length # [B,L]
            expanded_positions = source_positions.view(batch, 1, 1, length).expand(-1, heads, channels, -1)
            shifted = torch.gather(values, 3, expanded_positions)
            output = output + weights[:, delay_index].view(batch, 1, 1, 1) * shifted
        return output.permute(0, 3, 1, 2) # [B,L,H,E]

class DenseCore(nn.Module):
    def forward(self, queries, keys, values):   # each [B,L,H,E], used when auto-correlation is switched off
        scores = torch.einsum("blhe,bshe->bhls", queries, keys) / math.sqrt(queries.shape[-1])
        weights = torch.softmax(scores, dim = -1)
        return torch.einsum("bhls,bshe->blhe", weights, values)

def create_core(factor, use_autocorrelation):
    if use_autocorrelation:
        return AutoCorrelation(factor)
    return DenseCore()

class MixingLayer(nn.Module):
    def __init__(self, width, heads, core):
        super().__init__()
        self.heads = heads
        self.core = core
        self.query = nn.Linear(width, width)
        self.key = nn.Linear(width, width)
        self.value = nn.Linear(width, width)
        self.out = nn.Linear(width, width)

    def forward(self, queries, keys, values):
        batch, query_length, width = queries.shape
        source_length = keys.shape[1]
        head_width = width // self.heads
        query_heads = self.query(queries).view(batch, query_length, self.heads, head_width)
        key_heads = self.key(keys).view(batch, source_length, self.heads, head_width)
        value_heads = self.value(values).view(batch, source_length, self.heads, head_width)
        mixed = self.core(query_heads, key_heads, value_heads).reshape(batch, query_length, width)
        return self.out(mixed)

class SeasonalNorm(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)

    def forward(self, x):
        normed = self.norm(x)
        return normed - normed.mean(dim = 1, keepdim = True) # Removes the bias that layer norm adds to the seasonal part

class EncoderLayer(nn.Module):
    def __init__(self, width, kernel, use_decomposition, use_autocorrelation):
        super().__init__()
        self.attention = MixingLayer(width, head_count, create_core(delay_factor, use_autocorrelation))
        self.feed_forward = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(),
                                          nn.Dropout(dropout_rate), nn.Linear(2 * width, width))
        self.first_split = create_split(kernel, use_decomposition)
        self.second_split = create_split(kernel, use_decomposition)
        self.drop = nn.Dropout(dropout_rate)

    def forward(self, x):
        x, _ = self.first_split(x + self.drop(self.attention(x, x, x)))
        x, _ = self.second_split(x + self.drop(self.feed_forward(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, width, kernel, use_decomposition, use_autocorrelation, accumulate_trend):
        super().__init__()
        self.self_attention = MixingLayer(width, head_count, create_core(delay_factor, use_autocorrelation))
        self.cross_attention = MixingLayer(width, head_count, create_core(delay_factor, use_autocorrelation))
        self.feed_forward = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(),
                                          nn.Dropout(dropout_rate), nn.Linear(2 * width, width))
        self.first_split = create_split(kernel, use_decomposition)
        self.second_split = create_split(kernel, use_decomposition)
        self.third_split = create_split(kernel, use_decomposition)
        self.accumulate_trend = accumulate_trend
        self.trend_projection = None
        if accumulate_trend:
            self.trend_projection = nn.Conv1d(width, 1, 3, padding = 1, padding_mode = "circular", bias = False)
        self.drop = nn.Dropout(dropout_rate)

    def forward(self, x, memory, trend):
        x, trend_one = self.first_split(x + self.drop(self.self_attention(x, x, x)))
        x, trend_two = self.second_split(x + self.drop(self.cross_attention(x, memory, memory)))
        x, trend_three = self.third_split(x + self.drop(self.feed_forward(x)))
        if self.accumulate_trend:
            residual_trend = trend_one + trend_two + trend_three
            residual_trend = self.trend_projection(residual_trend.transpose(1, 2)).transpose(1, 2) # [B,T,1]
            trend = trend + residual_trend
        return x, trend

class Embedding(nn.Module):
    def __init__(self, covariate_count, width, concatenate):
        super().__init__()
        self.width = width
        self.concatenate = concatenate and covariate_count > 0
        input_channels = 1
        if self.concatenate:
            input_channels = 1 + covariate_count
        self.value_embed = nn.Conv1d(input_channels, width, 3, padding = 1, padding_mode = "circular", bias = False)
        self.covariate_embed = None
        if covariate_count > 0 and not self.concatenate:
            self.covariate_embed = nn.Linear(covariate_count, width, bias = False)
        self.drop = nn.Dropout(dropout_rate)

    def positional(self, length, current_device):
        positions = torch.arange(length, device = current_device).unsqueeze(1).float()
        divisors = torch.exp(torch.arange(0, self.width, 2, device = current_device).float() * (-math.log(10000) / self.width))
        encoding = torch.zeros(length, self.width, device = current_device)
        encoding[:, 0::2] = torch.sin(positions * divisors)
        encoding[:, 1::2] = torch.cos(positions * divisors)
        return encoding.unsqueeze(0)

    def forward(self, values, covariates):      # values [B,L,1], covariates [B,L,F] or None
        if self.concatenate:
            values = torch.cat([values, covariates], dim = 2)
        embedded = self.value_embed(values.transpose(1, 2)).transpose(1, 2)
        embedded = embedded + self.positional(values.shape[1], values.device)
        if self.covariate_embed is not None:
            embedded = embedded + self.covariate_embed(covariates)
        return self.drop(embedded)

class Autoformer(nn.Module):
    def __init__(self, config, encoder_covariate_count, decoder_covariate_count):
        super().__init__()
        width, encoder_layer_count, decoder_layer_count = size_settings[config["size_name"]]
        use_decomposition, use_autocorrelation, accumulate_trend = mechanism_settings[config["mechanism"]]
        concatenate = placement_settings[config["placement"]]["concatenate"]
        self.split = create_split(moving_average_kernel, use_decomposition)
        self.encoder_embedding = Embedding(encoder_covariate_count, width, concatenate)
        self.decoder_embedding = Embedding(decoder_covariate_count, width, concatenate)
        self.encoder_layers = nn.ModuleList([EncoderLayer(width, moving_average_kernel, use_decomposition,
            use_autocorrelation) for _ in range(encoder_layer_count)])
        self.encoder_norm = SeasonalNorm(width)
        self.decoder_layers = nn.ModuleList([DecoderLayer(width, moving_average_kernel, use_decomposition,
            use_autocorrelation, accumulate_trend) for _ in range(decoder_layer_count)])
        self.decoder_norm = SeasonalNorm(width)
        self.seasonal_projection = nn.Linear(width, 1)

    def forward(self, encoder_values, encoder_covariates, decoder_covariates):
        # encoder_values [B,S,1]; decoder covariates cover the label part and the horizon
        batch = encoder_values.shape[0]
        mean_values = encoder_values.mean(dim = 1, keepdim = True).repeat(1, prediction_length, 1)
        zeros = torch.zeros(batch, prediction_length, 1, device = encoder_values.device)
        seasonal_init, trend_init = self.split(encoder_values)
        trend = torch.cat([trend_init[:, -label_length:], mean_values], dim = 1) # Trend starts from the input mean
        seasonal = torch.cat([seasonal_init[:, -label_length:], zeros], dim = 1)
        memory = self.encoder_embedding(encoder_values, encoder_covariates)
        for layer in self.encoder_layers:
            memory = layer(memory)
        memory = self.encoder_norm(memory)
        decoded = self.decoder_embedding(seasonal, decoder_covariates)
        for layer in self.decoder_layers:
            decoded, trend = layer(decoded, memory, trend)
        seasonal_output = self.seasonal_projection(self.decoder_norm(decoded))
        return (seasonal_output + trend)[:, -prediction_length:, 0] # [B,H]

In [3]:
# Cell 3: data preparation, training, evaluation, ladder runner

def prepare_arrays(config, fit_end):
    # All statistics use only the data before fit_end
    target = target_values.copy()
    if "log1p" in config["loss_name"]:
        target = np.log1p(target) # This compresses the heavy tail before scaling
    target_mean = target[:fit_end].mean()
    target_std = target[:fit_end].std()
    scaled_target = (target - target_mean) / target_std
    columns = feature_groups[config["feature_group"]]
    covariates = external_values[:, np.array(columns, dtype = int)].copy()
    for position, column in enumerate(columns):
        if column in [3, 4, 5]:
            covariates[:, position] = np.log1p(covariates[:, position]) # These columns are skewed and non-negative
        if column < 6:
            column_mean = covariates[:fit_end, position].mean()
            column_std = covariates[:fit_end, position].std() + 1e-6
            covariates[:, position] = (covariates[:, position] - column_mean) / column_std
    return scaled_target.astype(np.float32), covariates.astype(np.float32), target_mean, target_std

def make_batch(scaled_target, covariates, origins, input_length, placement, include_target):
    settings = placement_settings[placement]
    encoder_values = torch.stack([scaled_target[origin - input_length:origin] for origin in origins]).unsqueeze(-1)
    encoder_covariates = None
    decoder_covariates = None
    if covariates.shape[1] > 0:
        if settings["encoder"]:
            encoder_covariates = torch.stack([covariates[origin - input_length:origin] for origin in origins])
        if settings["decoder"]:
            decoder_covariates = torch.stack([covariates[origin - label_length:origin + prediction_length] for origin in origins])
            if not settings["future"]:
                decoder_covariates[:, label_length:, :] = 0 # This hides the known values over the horizon
    future_target = None
    if include_target:
        future_target = torch.stack([scaled_target[origin:origin + prediction_length] for origin in origins])
    return encoder_values, encoder_covariates, decoder_covariates, future_target

def compute_loss(output, target, loss_name):
    if loss_name.endswith("mae"):
        return F.l1_loss(output, target)
    if "huber" in loss_name:
        return F.huber_loss(output, target, delta = 1)
    return F.mse_loss(output, target)

def compute_metrics(actual, predicted):
    error = actual - predicted
    mae = np.abs(error).mean()
    rmse = np.sqrt((error ** 2).mean())
    smape = 100 * np.mean(2 * np.abs(error) / (np.abs(actual) + np.abs(predicted) + 1e-8))
    return float(mae), float(rmse), float(smape)

def train_model(config, seed, fit_end, label):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    scaled_array, covariate_array, target_mean, target_std = prepare_arrays(config, fit_end)
    scaled_target = torch.tensor(scaled_array, device = device)
    covariates = torch.tensor(covariate_array, device = device)
    settings = placement_settings[config["placement"]]
    covariate_count = covariates.shape[1]
    encoder_count = covariate_count if settings["encoder"] else 0 # Unused sides get no embedding, so P stays honest
    decoder_count = covariate_count if settings["decoder"] else 0
    model = Autoformer(config, encoder_count, decoder_count).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    optimizer = torch.optim.AdamW(model.parameters(), lr = config["learning_rate"], weight_decay = 1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = config["maximum_epochs"])
    start_time = time.time()
    for epoch in range(1, config["maximum_epochs"] + 1):
        model.train()
        offset = random.randint(0, training_stride - 1) # A new offset each epoch covers all windows over time
        epoch_origins = list(range(maximum_input_length + offset, fit_end - prediction_length + 1, training_stride))
        random.shuffle(epoch_origins)
        total_loss = 0
        batch_count = 0
        for start in range(0, len(epoch_origins), batch_size):
            batch_origins = epoch_origins[start:start + batch_size]
            encoder_values, encoder_covariates, decoder_covariates, future_target = make_batch(
                scaled_target, covariates, batch_origins, config["input_length"], config["placement"], True)
            output = model(encoder_values, encoder_covariates, decoder_covariates)
            loss = compute_loss(output, future_target, config["loss_name"])
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            total_loss = total_loss + loss.item()
            batch_count = batch_count + 1
        scheduler.step()
        if epoch % 5 == 0 or epoch == config["maximum_epochs"]:
            print(f"    {label} epoch {epoch}/{config['maximum_epochs']} train loss {total_loss / batch_count:.4f} ({time.time() - start_time:.0f}s)", flush = True)
    return {"model": model, "scaled_target": scaled_target, "covariates": covariates, "target_mean": target_mean,
            "target_std": target_std, "parameters": parameter_count, "seconds": time.time() - start_time}

def forecast_origins(trained, config, origins):
    model = trained["model"]
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(origins), batch_size):
            batch_origins = origins[start:start + batch_size]
            encoder_values, encoder_covariates, decoder_covariates, _ = make_batch(
                trained["scaled_target"], trained["covariates"], batch_origins, config["input_length"], config["placement"], False)
            outputs.append(model(encoder_values, encoder_covariates, decoder_covariates).cpu().numpy())
    values = np.concatenate(outputs) * trained["target_std"] + trained["target_mean"]
    if "log1p" in config["loss_name"]:
        values = np.expm1(np.clip(values, None, 10)) # The clip avoids overflow when undoing log1p
    return np.clip(values, 0, None) # The target is non-negative

def evaluate_config(config, label):
    seed_rows = []
    for seed in seed_list:
        trained = train_model(config, seed, validation_start, label + " seed " + str(seed))
        predicted = forecast_origins(trained, config, validation_origins)
        mae, rmse, smape = compute_metrics(actual_blocks, predicted)
        composite = (mae + rmse + smape) / 3 # The agreed criterion: equal weight on MAE, RMSE and sMAPE
        seed_rows.append({"mae": mae, "rmse": rmse, "smape": smape, "composite": composite,
                          "parameters": trained["parameters"], "seconds": trained["seconds"]})
        print(f"  {label} seed {seed}: mae {mae:.3f} rmse {rmse:.3f} smape {smape:.2f} composite {composite:.3f}", flush = True)
        del trained
        torch.cuda.empty_cache()
    frame = pd.DataFrame(seed_rows)
    return {"composite": frame["composite"].mean(), "composite_std": frame["composite"].std(),
            "mae": frame["mae"].mean(), "rmse": frame["rmse"].mean(), "smape": frame["smape"].mean(),
            "parameters": int(frame["parameters"].iloc[0]), "epochs": config["maximum_epochs"],
            "seconds": frame["seconds"].mean()}

def run_ablation(number, title, key, values, base_config, fix_best):
    print("=" * 80, flush = True)
    print("Ablation " + str(number) + ": " + title + " (" + key + ")", flush = True)
    rows = []
    for value in values:
        config = dict(base_config)
        config[key] = value
        summary = evaluate_config(config, key + " = " + str(value))
        summary["value"] = str(value)
        rows.append(summary)
        print(f"  mean composite for {value}: {summary['composite']:.3f} (std {summary['composite_std']:.3f})", flush = True)
    table = pd.DataFrame(rows)[["value", "composite", "composite_std", "mae", "rmse", "smape", "parameters", "epochs", "seconds"]]
    table = table.sort_values("composite") # Best (lowest composite) first
    print("Results, best to worst:", flush = True)
    print(table.round(3).to_string(index = False), flush = True)
    table.to_csv(output_directory + "ablation_" + str(number) + ".csv", index = False)
    best_position = int(table.index[0])
    new_config = dict(base_config)
    if fix_best:
        new_config[key] = values[best_position]
        with open(output_directory + "best_config.json", "w") as file:
            json.dump(new_config, file)
        print("Fixed " + key + " = " + str(values[best_position]), flush = True)
    else:
        print("Comparison only: " + key + " stays " + str(base_config[key]), flush = True)
    return new_config

In [ ]:
# Cell 4: Ablation 1, optional file

best_config = run_ablation(1, "Optional file", "feature_group",
    ["none", "all", "continuous", "binary", "ABC", "DEF"], best_config, True)

In [ ]:
# Cell 5: Ablation 2, covariate placement

if best_config["feature_group"] == "none":
    print("Best feature group is none, so placement has no effect. Skipping ablation 2.", flush = True) # Nothing to place
else:
    best_config = run_ablation(2, "Covariate placement", "placement",
        ["encoder_only", "decoder_only", "both", "past_only_both", "both_concat"], best_config, True)

In [ ]:
# Cell 6: Ablation 8, model size

best_config = run_ablation(8, "Model size", "size_name",
    ["tiny", "small", "medium", "large", "xlarge"], best_config, True)

In [ ]:
# Cell 7: Ablation 11, epochs

best_config = run_ablation(11, "Training epochs", "maximum_epochs",
    [2, 5, 10, 25, 40], best_config, True)

In [ ]:
# Cell 8: Ablation 4, Autoformer mechanisms (comparison only)

# The assignment requires both mechanisms in the submitted model, so this is comparison only and nothing is fixed
best_config = run_ablation(4, "Autoformer mechanisms", "mechanism",
    ["full", "no_decomposition", "no_autocorrelation", "neither", "no_trend_accumulation"], best_config, False)

In [ ]:
# Cell 9: Ablation 9, loss

best_config = run_ablation(9, "Loss function", "loss_name",
    ["mse", "mae", "huber", "log1p_mse", "log1p_huber"], best_config, True)

In [ ]:
# Cell 10: Extra ablation A, input length

best_config = run_ablation("A", "Input length", "input_length",
    [96, 168, 336, 672, 1008], best_config, True)

In [ ]:
# Cell 11: Extra ablation B, learning rate

best_config = run_ablation("B", "Learning rate", "learning_rate",
    [1e-4, 3e-4, 1e-3, 3e-3, 1e-2], best_config, True)
print("Final ladder configuration:", best_config, flush = True)

In [8]:
# Cell 12: Leaderboard output

# This cell needs only Cells 1 to 3 to have been run, not the ablations. It reads best_config.json or your manual config.

config_source = "manual" # Use "ablation" to read the saved ladder result, or "manual" to use manual_config below
manual_config = {
    "feature_group": "all",
    "placement": "both",
    "size_name": "small",
    "maximum_epochs": 20,
    "mechanism": "full",
    "loss_name": "mse",
    "input_length": 1008,
    "learning_rate": 1e-2,
}
final_seed_list = [0] # More than one seed ensembles the models, and then P and E are summed

if config_source == "ablation":
    with open(output_directory + "best_config.json") as file:
        final_config = json.load(file)
else:
    final_config = manual_config
print("Final config:", final_config, flush = True)

forecast_list = []
total_parameters = 0
total_epochs = 0
for seed in final_seed_list:
    trained = train_model(final_config, seed, train_length, "final seed " + str(seed)) # Fits on the full series
    forecast = forecast_origins(trained, final_config, [train_length])[0] # Forecast for time_idx 43657 to 43824
    forecast_list.append(forecast)
    total_parameters = total_parameters + trained["parameters"]
    total_epochs = total_epochs + final_config["maximum_epochs"]
    del trained
    torch.cuda.empty_cache()

final_forecast = np.mean(forecast_list, axis = 0)
assert len(final_forecast) == 168 and not np.isnan(final_forecast).any()
submission_text = ",".join(f"{value:.4f}" for value in final_forecast)
with open(output_directory + "submission.txt", "w") as file:
    file.write(submission_text)

print("Parameters (P):", total_parameters, flush = True)
print("Epochs (E):", total_epochs, flush = True)
print("Predictions (paste into the leaderboard):", flush = True)
print(submission_text, flush = True)

Final config: {'feature_group': 'all', 'placement': 'both', 'size_name': 'small', 'maximum_epochs': 20, 'mechanism': 'full', 'loss_name': 'mse', 'input_length': 1008, 'learning_rate': 0.01}
    final seed 0 epoch 5/20 train loss 0.5066 (27s)
    final seed 0 epoch 10/20 train loss 0.4605 (54s)
    final seed 0 epoch 15/20 train loss 0.4244 (80s)
    final seed 0 epoch 20/20 train loss 0.4110 (107s)
Parameters (P): 5953
Epochs (E): 20
Predictions (paste into the leaderboard):
37.2313,36.4395,31.2592,36.4522,23.5137,28.6703,38.6200,41.2963,51.2945,46.4623,39.0520,39.2383,43.9703,45.8825,53.0547,57.7155,70.2132,79.8570,98.6453,99.6189,118.5819,122.5392,130.2075,137.1397,135.1259,133.7845,131.3006,143.2612,150.3100,157.2784,165.9086,179.0943,177.6342,183.1346,174.9753,173.0562,179.6273,186.5463,193.2953,193.4897,203.9769,228.4583,237.2502,230.9246,236.0148,224.0453,227.4260,229.7710,224.6427,206.9822,211.2777,209.0215,211.7934,215.9642,213.7989,204.2601,205.0201,205.4006,207.3281,208.0252,

In [ ]:
manual_config = {
    "feature_group": "all",
    "placement": "both",
    "size_name": "medium",
    "maximum_epochs": 25,
    "mechanism": "full",
    "loss_name": "mse",
    "input_length": 1008,
    "learning_rate": 1e-2,
}

Final config: {'feature_group': 'all', 'placement': 'both', 'size_name': 'small', 'maximum_epochs': 10, 'mechanism': 'full', 'loss_name': 'mse', 'input_length': 1008, 'learning_rate': 0.01}
    final seed 0 epoch 5/10 train loss 0.4970 (29s)
    final seed 0 epoch 10/10 train loss 0.4410 (55s)
Parameters (P): 5953
Epochs (E): 10
Predictions (paste into the leaderboard):
2.8990,9.7723,16.0937,33.0255,30.2574,35.5261,43.9747,45.3750,45.4125,35.5179,33.5460,29.1534,24.6757,24.8656,38.6613,49.6036,59.6169,63.8285,68.6072,73.6750,90.0915,105.3760,110.6947,111.8670,120.5847,130.8040,125.7638,123.9160,126.2030,129.8019,127.9189,130.8531,136.5186,145.4000,145.2273,150.9243,144.9676,140.3395,141.3469,147.1557,152.2959,164.5931,182.8222,172.9647,181.6214,172.1471,161.0344,154.5239,161.7743,146.7290,132.8295,121.4019,122.3874,126.3489,130.6662,125.0578,120.6901,118.1137,115.4419,108.3506,109.3747,134.7670,135.2666,141.5622,142.0268,143.7725,147.9768,157.8175,165.8097,157.8300,151.0836,146.2651,151.6062,151.5534,143.7415,133.5580,131.4498,129.4062,125.4030,137.4110,151.2238,145.3081,130.5795,113.8514,116.0447,108.9209,96.8961,90.7523,84.5259,79.6317,78.9173,78.7499,101.2872,99.6674,100.9620,105.0141,107.6713,116.5041,125.7595,131.5794,125.2914,118.5651,117.2451,113.4855,119.2335,123.6271,101.1054,95.7694,89.1080,77.7672,71.4301,68.9208,68.1475,66.8152,78.7331,76.7319,89.7346,105.4054,117.1573,108.1036,96.2392,97.0150,98.3081,87.8858,80.6208,62.0754,61.4984,82.0183,64.2324,63.5820,62.4949,71.1214,60.0597,51.6056,51.7844,56.5834,73.2510,72.6992,84.4448,86.9958,82.5130,75.4588,46.9053,39.1377,40.6270,30.0810,23.1691,14.2573,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000

In [ ]:
manual_config = {
    "feature_group": "none",
    "placement": "both",
    "size_name": "medium",
    "maximum_epochs": 10,
    "mechanism": "full",
    "loss_name": "mse",
    "input_length": 1008,
    "learning_rate": 1e-2,
}

Final config: {'feature_group': 'none', 'placement': 'both', 'size_name': 'medium', 'maximum_epochs': 10, 'mechanism': 'full', 'loss_name': 'mse', 'input_length': 1008, 'learning_rate': 0.01}
    final seed 0 epoch 5/10 train loss 1.0184 (38s)
    final seed 0 epoch 10/10 train loss 1.0024 (75s)
Parameters (P): 21505
Epochs (E): 10
Predictions (paste into the leaderboard):
100.1135,104.1247,101.1949,99.7558,102.4514,108.0903,109.7280,107.5189,107.8200,110.3025,110.4743,107.8267,107.6813,105.9180,105.9511,105.7262,104.1874,103.1356,103.5186,108.0402,113.6095,116.0451,113.7869,111.3770,109.8479,107.6463,107.5974,105.6981,106.9092,107.7828,105.6431,100.8658,100.6221,102.6432,100.9873,99.2900,98.3996,99.1073,99.9977,98.1337,97.4132,97.8277,100.2800,98.9025,97.8503,100.5757,100.1985,103.1305,107.7739,110.6912,115.0099,120.1308,121.5999,117.4611,111.0431,109.1919,109.9317,110.5423,115.3283,115.7615,113.9207,115.8989,118.8387,119.8863,121.7259,123.1505,122.4572,120.9967,116.3006,120.2760,124.0237,118.0190,109.3099,103.9009,102.9898,105.6501,107.0477,106.9674,105.6084,106.2392,100.5323,97.6894,98.9651,99.9857,99.2442,102.1668,102.9564,105.3225,111.0510,109.1498,109.2193,110.0016,110.9987,111.6777,108.8207,104.1035,104.1819,105.5679,106.5837,107.9224,112.0296,111.7480,107.2468,106.5215,99.7341,99.5200,100.1173,102.0447,102.8644,102.8803,104.0557,104.8045,105.3896,106.3530,105.7415,109.8798,109.2159,112.0000,110.7065,109.5498,109.4494,108.9971,110.7954,110.0867,109.9413,108.8035,104.1724,106.4885,109.6860,110.5895,109.9228,105.5392,102.8038,104.7368,105.5711,107.0353,108.6434,108.5033,111.8839,118.2574,125.6921,130.0895,135.2996,134.4395,134.3225,136.1935,140.9746,143.5627,146.9783,149.9489,154.5533,157.0155,156.2995,155.0104,150.3293,145.1247,147.1176,147.3844,146.6676,146.2161,145.6921,146.1012,148.3447,149.2237,145.8699,141.1790,135.7448,146.8216

In [ ]:
manual_config = { # BEST
    "feature_group": "all",
    "placement": "both",
    "size_name": "small",
    "maximum_epochs": 20,
    "mechanism": "full",
    "loss_name": "mse",
    "input_length": 1008,
    "learning_rate": 1e-2,
}

Final config: {'feature_group': 'all', 'placement': 'both', 'size_name': 'small', 'maximum_epochs': 20, 'mechanism': 'full', 'loss_name': 'mse', 'input_length': 1008, 'learning_rate': 0.01}
    final seed 0 epoch 5/20 train loss 0.5066 (27s)
    final seed 0 epoch 10/20 train loss 0.4605 (54s)
    final seed 0 epoch 15/20 train loss 0.4244 (80s)
    final seed 0 epoch 20/20 train loss 0.4110 (107s)
Parameters (P): 5953
Epochs (E): 20
Predictions (paste into the leaderboard):
37.2313,36.4395,31.2592,36.4522,23.5137,28.6703,38.6200,41.2963,51.2945,46.4623,39.0520,39.2383,43.9703,45.8825,53.0547,57.7155,70.2132,79.8570,98.6453,99.6189,118.5819,122.5392,130.2075,137.1397,135.1259,133.7845,131.3006,143.2612,150.3100,157.2784,165.9086,179.0943,177.6342,183.1346,174.9753,173.0562,179.6273,186.5463,193.2953,193.4897,203.9769,228.4583,237.2502,230.9246,236.0148,224.0453,227.4260,229.7710,224.6427,206.9822,211.2777,209.0215,211.7934,215.9642,213.7989,204.2601,205.0201,205.4006,207.3281,208.0252,209.9292,235.9144,229.1413,226.0610,226.1554,235.9547,251.6070,262.7991,271.5468,271.5540,265.5684,251.1947,245.6451,239.0711,230.4400,211.6139,206.0232,197.8151,194.9008,201.6877,209.5348,198.5633,171.5101,141.5795,145.5105,135.7378,130.4528,130.9795,133.8554,131.6882,124.0259,117.4103,126.7035,120.6479,126.3352,143.5850,147.8428,153.4718,162.7148,167.2449,163.2289,157.7805,159.4688,146.0506,147.1925,148.2777,123.2512,121.2912,134.7300,135.0439,132.7178,130.7245,127.8280,126.1882,143.0335,149.7391,163.2592,175.1943,184.0809,180.1740,166.9995,164.8228,151.1082,137.7380,140.7035,128.5093,130.8486,132.6837,142.3279,151.1172,135.1613,133.4441,131.1257,131.1110,129.3035,130.9427,153.0295,158.5176,169.1005,173.5029,175.5683,162.4089,128.9848,116.9322,124.1292,117.9006,104.8740,94.3000,78.6899,62.2614,58.0631,58.2241,53.2203,43.7862,44.2698,53.5986,47.7336,42.9541,35.1725,38.5951,53.5996,52.2373,56.2848,48.2324,48.3772,38.9061,36.5469,21.8819